In [ ]:
# Step 2: Run the script with the correct path - note the quotes around the path to handle spaces
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/colab_setup.py"

In [ ]:
!pip install -q tqdm matplotlib tensorflow
!pip install opencv-python-headless
import cv2
!pip install yolo
import yolo
import tensorflow as tf
import os
import glob
import json
import sys
import shutil
import re
import random
import xml.etree.ElementTree as ET
import json
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path

In [ ]:
# Add the project root to Python's path
project_root = '/content/drive/MyDrive/Colab Notebooks/Pawnder'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now try to import from config
try:
    from config import DATA_DIRS, ensure_directories
except ModuleNotFoundError:
    print("Could not import config module. Adjusting path...")
    # Try with full path
    sys.path.insert(0, os.path.abspath(project_root))
    from config import DATA_DIRS, ensure_directories

# Make sure directories exist
ensure_directories()

def process_data():
    """Run the complete data processing pipeline"""
    print("Starting Pawnder data processing pipeline...")

In [ ]:

# Set random seed for reproducibility
random.seed(42)

# Define key paths
base_dir = "/content/drive/MyDrive/Colab Notebooks/Pawnder"
videos_dir = os.path.join(base_dir, "Data/Raw/personal_dataset/videos")
personal_images_dir = os.path.join(base_dir, "Data/Raw/personal_dataset/images")
stanford_dir = os.path.join(base_dir, "Data/Raw/stanford_dog_pose")
stanford_images_dir = os.path.join(stanford_dir, "Images")
emotions_file = os.path.join(base_dir, "Data/interim/emotions_only.json")

# Output directories
output_dir = os.path.join(base_dir, "Data/processed")
combined_frames_dir = os.path.join(output_dir, "all_frames")
os.makedirs(combined_frames_dir, exist_ok=True)

# Emotion mapping from old to new categories
EMOTION_MAPPING = {
    "Happy or Playful": "Happy/Playful",
    "Relaxed": "Relaxed",
    "Submissive": "Submissive/Appeasement",
    "Excited": "Happy/Playful",
    "Drowsy or Bored": "Relaxed",
    "Curious or Confused": "Curiosity/Alertness",
    "Confident or Alert": "Curiosity/Alertness",
    "Jealous": "Stressed",
    "Stressed": "Stressed",
    "Frustrated": "Stressed",
    "Unsure or Uneasy": "Fearful/Anxious",
    "Possessive, Territorial, Dominant": "Aggressive/Threatening",
    "Fear or Aggression": "Fearful/Anxious",
    "Pain": "Stressed"
}

# Define emotion classes
CLASS_NAMES = ["Happy/Playful", "Relaxed", "Submissive/Appeasement",
               "Curiosity/Alertness", "Stressed", "Fearful/Anxious",
               "Aggressive/Threatening"]

# Safe class names for directories
SAFE_CLASS_NAMES = [emotion.replace("/", "_").replace("\\", "_") for emotion in CLASS_NAMES]
CLASS_MAP = dict(zip(CLASS_NAMES, SAFE_CLASS_NAMES))

# Create output directories
for split in ["all_by_class", "train_by_class", "val_by_class", "test_by_class"]:
    split_dir = os.path.join(output_dir, split)
    os.makedirs(split_dir, exist_ok=True)
    for safe_name in SAFE_CLASS_NAMES:
        os.makedirs(os.path.join(split_dir, safe_name), exist_ok=True)
    os.makedirs(os.path.join(split_dir, "unknown"), exist_ok=True)

def parse_xml_annotation(xml_file):
    """Parse CVAT XML annotations"""
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Get video name from task source
        video_name = None
        for meta in root.findall('.//meta'):
            for task in meta.findall('.//task'):
                for source in task.findall('.//source'):
                    video_name = source.text
                    if video_name:
                        # Remove extension if present
                        video_name = os.path.splitext(video_name)[0]

        if not video_name:
            video_name = os.path.basename(os.path.dirname(xml_file))

        # Extract annotations
        annotations = {}

        for track in root.findall('.//track'):
            for box in track.findall('.//box'):
                frame_num = int(box.get('frame', 0))
                frame_id = f"frame_{frame_num:06d}"

                # Look for Primary Emotion attribute
                emotion = None
                for attr in box.findall('.//attribute'):
                    name = attr.get('name')
                    if name == "Primary Emotion":
                        emotion = attr.text
                        break

                if emotion:
                    # Map to standardized emotion if needed
                    if emotion in EMOTION_MAPPING:
                        emotion = EMOTION_MAPPING[emotion]

                    annotations[frame_id] = {
                        "emotions": {"primary_emotion": emotion},
                        "video_name": video_name,
                        "frame": frame_num,
                        "original_format": "xml"
                    }

        return annotations, video_name

    except Exception as e:
        print(f"Error parsing XML file {xml_file}: {str(e)}")
        return {}, None

def process_video_folder(video_folder):
    """Process a video folder with images and annotations"""
    folder_name = os.path.basename(video_folder)
    print(f"\nProcessing video folder: {folder_name}")

    # Find any XML file in this folder
    xml_files = glob.glob(os.path.join(video_folder, "*.xml"))
    if not xml_files:
        print(f"  No XML files found in {video_folder}")
        return 0, {}

    # Use the first XML file
    xml_file = xml_files[0]
    print(f"  Using XML file: {os.path.basename(xml_file)}")

    # Parse annotations
    annotations, video_name = parse_xml_annotation(xml_file)
    if not video_name:
        video_name = folder_name

    # Generate video ID
    video_id = ''.join(e for e in video_name if e.isalnum())[:8]

    # Look for images directory
    images_dir = os.path.join(video_folder, "images")
    if not os.path.exists(images_dir):
        print(f"  Images directory not found: {images_dir}")
        return 0, {}

    # Count annotations by emotion
    emotion_counts = defaultdict(int)
    for frame_id, data in annotations.items():
        emotion = data["emotions"]["primary_emotion"]
        emotion_counts[emotion] += 1

    print(f"  Found {len(annotations)} annotated frames")
    print("  Emotion distribution:")
    for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"    {emotion}: {count}")

    # Get all files in the images directory
    all_files = os.listdir(images_dir)
    print(f"  Found {len(all_files)} files in images directory")

    # Create filename mapping for quick lookup
    filename_map = {}
    for filename in all_files:
        match = re.search(r'frame_0*(\d+)', filename.lower())
        if match:
            frame_num = int(match.group(1))
            if frame_num not in filename_map:
                filename_map[frame_num] = []
            filename_map[frame_num].append(filename)

    # Process each file
    processed_frames = {}
    processed_count = 0
    missing_count = 0

    for frame_id, annotation in tqdm(annotations.items(), desc=f"Processing {folder_name}", leave=False):
        # Extract frame number
        match = re.search(r'frame_0*(\d+)', frame_id)
        if not match:
            continue

        frame_num = int(match.group(1))

        # Find matching file
        if frame_num in filename_map and filename_map[frame_num]:
            src_filename = filename_map[frame_num][0]
            src_path = os.path.join(images_dir, src_filename)

            # Get emotion and create safe version
            emotion = annotation["emotions"]["primary_emotion"]
            safe_emotion = emotion.replace("/", "_").replace("\\", "_")

            # Create new filename with consistent format
            new_filename = f"video_{video_id}_frame_{frame_num:06d}{os.path.splitext(src_filename)[1]}"
            dst_path = os.path.join(combined_frames_dir, new_filename)

            # Copy to combined directory
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(src_path, dst_path)

            # Copy to class directory
            class_dir = os.path.join(output_dir, "all_by_class", safe_emotion)
            class_path = os.path.join(class_dir, new_filename)
            os.makedirs(os.path.dirname(class_path), exist_ok=True)
            shutil.copy2(src_path, class_path)

            # Add to processed frames
            processed_frames[new_filename] = {
                "emotions": {"primary_emotion": emotion},
                "video_name": video_name,
                "video_id": video_id,
                "frame_id": frame_id,
                "original_path": src_path,
                "source": "video_frames"
            }

            processed_count += 1
        else:
            missing_count += 1

    print(f"  Processed {processed_count} frames, {missing_count} frames missing")
    return processed_count, processed_frames

def process_personal_images():
    """Process personal dataset images"""
    print("\nProcessing Personal Dataset Images")

    # Check if emotions_only.json exists
    if not os.path.exists(emotions_file):
        print(f"  Error: emotions_only.json not found at {emotions_file}")
        return 0, {}

    # Load the emotions file
    try:
        with open(emotions_file, 'r') as f:
            all_annotations = json.load(f)
        print(f"  Loaded {len(all_annotations)} annotations from emotions_only.json")
    except Exception as e:
        print(f"  Error loading emotions file: {str(e)}")
        return 0, {}

    # Check if personal_images_dir exists
    if not os.path.exists(personal_images_dir):
        print(f"  Error: Personal images directory not found at {personal_images_dir}")
        return 0, {}

    # Build a map of all personal images
    personal_images = {}
    image_files = []

    # Look for images in the root and subdirectories
    for root, dirs, files in os.walk(personal_images_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                file_path = os.path.join(root, file)
                image_files.append(file_path)
                # Add with basename as key
                personal_images[file] = file_path
                # Also add without extension
                name_without_ext = os.path.splitext(file)[0]
                personal_images[name_without_ext] = file_path

    print(f"  Found {len(image_files)} total personal images")

    # Find personal images in annotations
    processed_frames = {}
    processed_count = 0

    for img_id, annotation in tqdm(all_annotations.items(), desc="Processing personal annotations"):
        # Skip if this doesn't have an emotion
        if "emotions" not in annotation or "primary_emotion" not in annotation["emotions"]:
            continue

        # Get the filename
        filename = os.path.basename(img_id)
        basename = os.path.splitext(filename)[0]

        # Check if we have this image
        image_path = None
        if filename in personal_images:
            image_path = personal_images[filename]
        elif basename in personal_images:
            image_path = personal_images[basename]

        # Skip if it's a Stanford image
        is_stanford = re.match(r'n\d+_\d+', basename) is not None
        if is_stanford:
            continue

        # Skip if it's a video frame (already processed)
        is_video_frame = "frame_" in basename
        if is_video_frame:
            continue

        if image_path:
            # Get emotion and create safe version
            emotion = annotation["emotions"]["primary_emotion"]
            safe_emotion = emotion.replace("/", "_").replace("\\", "_")

            # Create new filename
            new_filename = f"personal_{basename}{os.path.splitext(image_path)[1]}"
            dst_path = os.path.join(combined_frames_dir, new_filename)

            # Copy to combined directory
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            try:
                shutil.copy2(image_path, dst_path)

                # Copy to class directory
                class_dir = os.path.join(output_dir, "all_by_class", safe_emotion)
                class_path = os.path.join(class_dir, new_filename)
                os.makedirs(os.path.dirname(class_path), exist_ok=True)
                shutil.copy2(image_path, class_path)

                # Add to processed frames
                processed_frames[new_filename] = {
                    "emotions": {"primary_emotion": emotion},
                    "original_id": img_id,
                    "original_path": image_path,
                    "source": "personal"
                }

                processed_count += 1

                if processed_count % 100 == 0:
                    print(f"  Processed {processed_count} personal images")

            except Exception as e:
                print(f"  Error copying {image_path}: {str(e)}")

    print(f"  Processed {processed_count} personal images with emotion annotations")

    # Count by emotion
    emotion_counts = defaultdict(int)
    for _, data in processed_frames.items():
        emotion = data["emotions"]["primary_emotion"]
        emotion_counts[emotion] += 1

    print("  Personal dataset emotion distribution:")
    for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"    {emotion}: {count}")

    return processed_count, processed_frames

def process_stanford_dataset():
    """Process Stanford dog dataset images"""
    print("\nProcessing Stanford Dog Dataset")

    # Check if emotions_only.json exists
    if not os.path.exists(emotions_file):
        print(f"  Error: emotions_only.json not found at {emotions_file}")
        return 0, {}

    # Load the emotions file
    try:
        with open(emotions_file, 'r') as f:
            all_annotations = json.load(f)
        print(f"  Loaded {len(all_annotations)} annotations from emotions_only.json")
    except Exception as e:
        print(f"  Error loading emotions file: {str(e)}")
        return 0, {}

    # Check if stanford_images_dir exists
    if not os.path.exists(stanford_images_dir):
        print(f"  Error: Stanford images directory not found at {stanford_images_dir}")
        return 0, {}

    # Find all stanford breed directories
    breed_dirs = []
    for item in os.listdir(stanford_images_dir):
        breed_path = os.path.join(stanford_images_dir, item)
        if os.path.isdir(breed_path) and item.startswith('n'):  # Stanford uses n* format for breed directories
            breed_dirs.append(breed_path)

    print(f"  Found {len(breed_dirs)} breed directories")

    # Build a map of all Stanford images
    stanford_images = {}
    for breed_dir in breed_dirs:
        breed_name = os.path.basename(breed_dir)
        image_files = glob.glob(os.path.join(breed_dir, "*.jpg")) + \
                     glob.glob(os.path.join(breed_dir, "*.png"))

        for image_path in image_files:
            image_name = os.path.basename(image_path)
            stanford_images[image_name] = image_path
            # Also add without extension
            name_without_ext = os.path.splitext(image_name)[0]
            stanford_images[name_without_ext] = image_path

    print(f"  Found {len(stanford_images)} total Stanford dog images")

    # Find Stanford images in annotations
    processed_frames = {}
    processed_count = 0

    for img_id, annotation in tqdm(all_annotations.items(), desc="Processing Stanford annotations"):
        # Skip if this doesn't have an emotion
        if "emotions" not in annotation or "primary_emotion" not in annotation["emotions"]:
            continue

        # Check if this is likely a Stanford image
        is_stanford = False
        filename = os.path.basename(img_id)
        basename = os.path.splitext(filename)[0]

        # Stanford images often have format like n02085620_10074.jpg
        if re.match(r'n\d+_\d+', basename) or filename in stanford_images or basename in stanford_images:
            is_stanford = True

        if not is_stanford:
            continue

        # Find the image path
        image_path = None
        if filename in stanford_images:
            image_path = stanford_images[filename]
        elif basename in stanford_images:
            image_path = stanford_images[basename]

        if image_path:
            # Get emotion and create safe version
            emotion = annotation["emotions"]["primary_emotion"]
            safe_emotion = emotion.replace("/", "_").replace("\\", "_")

            # Create new filename
            new_filename = f"stanford_{basename}{os.path.splitext(image_path)[1]}"
            dst_path = os.path.join(combined_frames_dir, new_filename)

            # Copy to combined directory
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            try:
                shutil.copy2(image_path, dst_path)

                # Copy to class directory
                class_dir = os.path.join(output_dir, "all_by_class", safe_emotion)
                class_path = os.path.join(class_dir, new_filename)
                os.makedirs(os.path.dirname(class_path), exist_ok=True)
                shutil.copy2(image_path, class_path)

                # Add to processed frames
                processed_frames[new_filename] = {
                    "emotions": {"primary_emotion": emotion},
                    "original_id": img_id,
                    "original_path": image_path,
                    "source": "stanford"
                }

                processed_count += 1

                if processed_count % 100 == 0:
                    print(f"  Processed {processed_count} Stanford images")

            except Exception as e:
                print(f"  Error copying {image_path}: {str(e)}")

    print(f"  Processed {processed_count} Stanford images with emotion annotations")

    # Count by emotion
    emotion_counts = defaultdict(int)
    for _, data in processed_frames.items():
        emotion = data["emotions"]["primary_emotion"]
        emotion_counts[emotion] += 1

    print("  Stanford emotion distribution:")
    for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"    {emotion}: {count}")

    return processed_count, processed_frames

# MAIN PROCESSING PIPELINE

# Step 1: Process video folders
print("Step 1: Processing video folders...")
video_folders = []
for item in os.listdir(videos_dir):
    folder_path = os.path.join(videos_dir, item)
    if os.path.isdir(folder_path):
        video_folders.append(folder_path)

print(f"Found {len(video_folders)} video folders")

video_frames = {}
total_video_frames = 0

for video_folder in tqdm(video_folders, desc="Processing video folders"):
    count, frames = process_video_folder(video_folder)
    total_video_frames += count
    video_frames.update(frames)

print(f"\nProcessed {total_video_frames} video frames from {len(video_folders)} folders")

# Step 2: Process Stanford dataset
print("\nStep 2: Processing Stanford dataset...")
stanford_count, stanford_frames = process_stanford_dataset()

# Step 3: Process personal images
print("\nStep 3: Processing personal dataset images...")
personal_count, personal_frames = process_personal_images()

# Step 4: Combine all annotations
all_frames = {}
all_frames.update(video_frames)
all_frames.update(stanford_frames)
all_frames.update(personal_frames)

total_frames = len(all_frames)
print(f"\nTotal processed frames: {total_frames}")
print(f"  - {len(video_frames)} video frames")
print(f"  - {len(stanford_frames)} Stanford images")
print(f"  - {len(personal_frames)} personal images")

# Save combined annotations
annotations_file = os.path.join(output_dir, "combined_annotations.json")
with open(annotations_file, 'w') as f:
    json.dump(all_frames, f, indent=2)

print(f"Saved annotations to: {annotations_file}")

# Create train/val/test splits
print("\nStep 5: Creating train/val/test splits...")

# Group by emotion
frames_by_emotion = defaultdict(list)
for filename, data in all_frames.items():
    emotion = data["emotions"]["primary_emotion"]
    safe_emotion = emotion.replace("/", "_").replace("\\", "_")
    frames_by_emotion[safe_emotion].append((filename, data))

print("Emotion distribution:")
for emotion, frames in sorted(frames_by_emotion.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"  {emotion}: {len(frames)} frames")

# Split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# Perform stratified split
train_files = []
val_files = []
test_files = []

for emotion, files in frames_by_emotion.items():
    random.shuffle(files)
    n_files = len(files)
    n_train = int(n_files * train_ratio)
    n_val = int(n_files * val_ratio)

    train_files.extend(files[:n_train])
    val_files.extend(files[n_train:n_train+n_val])
    test_files.extend(files[n_train+n_val:])

# Copy files to split directories
split_counts = defaultdict(lambda: defaultdict(int))

for file_list, split_dir, split_name in [
    (train_files, os.path.join(output_dir, "train_by_class"), "train"),
    (val_files, os.path.join(output_dir, "val_by_class"), "validation"),
    (test_files, os.path.join(output_dir, "test_by_class"), "test")
]:
    print(f"\nCreating {split_name} split with {len(file_list)} files...")

    for filename, data in tqdm(file_list, desc=f"Processing {split_name} split"):
        emotion = data["emotions"]["primary_emotion"]
        safe_emotion = emotion.replace("/", "_").replace("\\", "_")

        src_path = os.path.join(combined_frames_dir, filename)
        dst_path = os.path.join(split_dir, safe_emotion, filename)

        if os.path.exists(src_path):
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(src_path, dst_path)
            split_counts[split_name][safe_emotion] += 1

# Print final statistics
print("\nFinal Dataset Statistics:")
print(f"Total images: {total_frames}")

# Print split statistics
for split_name in ["train", "validation", "test"]:
    emotion_counts = split_counts[split_name]
    total = sum(emotion_counts.values())

    print(f"\n{split_name.capitalize()} split:")
    print(f"  Total: {total} images")
    print("  Emotion distribution:")
    for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
        percent = count / total * 100 if total > 0 else 0
        print(f"    {emotion}: {count} ({percent:.1f}%)")

print("\nDataset preparation complete!")

# NOT SURE IF NEED THESE

In [ ]:
# Step 1: Organize all images from various sources
import os
import json
import shutil
from config import DATA_DIRS

# Create destination directory
dest_dir = os.path.join(DATA_DIRS['processed'], 'all_images')
os.makedirs(dest_dir, exist_ok=True)

# Define image source locations
image_sources = [
    os.path.join(DATA_DIRS['personal_dataset'], 'images'),
    os.path.join(DATA_DIRS['stanford_original'], 'train', 'images'),
    os.path.join(DATA_DIRS['stanford_original'], 'val', 'images'),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/Pawnder/images_backup"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/Dog Emotion Analyzer/raw_data/AWS S3download/Dog Emotion Videos/test"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/Dog Emotion Analyzer/raw_data/AWS S3download/Dog Emotion Videos/train"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/Dog Emotion Analyzer/raw_data/AWS S3download/Dog Emotion Videos/valid"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/Dog Emotion Analyzer/raw_data/AWS S3download/Dog Emotion Videos"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/Dog Emotion Analyzer/raw_data/AWS S3download/images"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/Dog Emotion Analyzer/raw video/video_frames"),
    os.path.join("/content/drive/MyDrive/Colab Notebooks/Dog Emotion Analyzer/raw_data/AWS S3download/val images/images")
]

# Copy images from all sources
total_copied = 0
for source_dir in image_sources:
    if not os.path.exists(source_dir):
        print(f"Source directory not found: {source_dir}")
        continue

    print(f"Copying images from: {source_dir}")
    copied = 0
    for filename in os.listdir(source_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            src_path = os.path.join(source_dir, filename)
            dst_path = os.path.join(dest_dir, filename)
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)
                copied += 1

    print(f"  Copied {copied} images")
    total_copied += copied

print(f"✅ Total images organized: {total_copied}")
print(f"Images directory: {dest_dir}")

In [ ]:
# Step 2: Fix image paths with prefixes
import os
import json
import shutil
from config import DATA_DIRS

# Load merged annotations
merged_file = os.path.join(DATA_DIRS['merged_annotations'], 'merged_annotations.json')
with open(merged_file, 'r') as f:
    annotations = json.load(f)

# Define source and destination directories
source_dir = os.path.join(DATA_DIRS['personal_dataset'], 'images')
dest_dir = os.path.join(DATA_DIRS['processed'], 'all_images')

# Counter for newly found images
newly_found = 0

# Check for annotations with prefixes
for img_id, data in annotations.items():
    if img_id.startswith(("Dog Emotion Videos/", "Personal Data Set/")):
        # Extract the filename part from the path
        filename = os.path.basename(img_id)

        # Check if this file exists in the source directory
        for ext in ['.jpg', '.jpeg', '.png', '']:
            # If filename already has extension, don't add another
            if any(filename.lower().endswith(e) for e in ['.jpg', '.jpeg', '.png']):
                src_path = os.path.join(source_dir, filename)
            else:
                src_path = os.path.join(source_dir, f"{filename}{ext}")

            if os.path.exists(src_path):
                # Found the image! Copy it to destination
                dest_path = os.path.join(dest_dir, os.path.basename(src_path))
                if not os.path.exists(dest_path):
                    shutil.copy2(src_path, dest_path)
                    newly_found += 1
                break

print(f"✅ Found and copied {newly_found} additional images with path prefixes")

In [ ]:
# Step 3: Create filtered annotations
import os
import json
from config import DATA_DIRS

# Load merged annotations
merged_file = os.path.join(DATA_DIRS['merged_annotations'], 'merged_annotations.json')
with open(merged_file, 'r') as f:
    annotations = json.load(f)

# Define destination paths
images_dir = os.path.join(DATA_DIRS['processed'], 'all_images')
filtered_file = os.path.join(DATA_DIRS['merged_annotations'], 'filtered_annotations.json')

# Filter annotations based on available images
filtered_annotations = {}
found_count = 0

for img_id, data in annotations.items():
    # For IDs with path prefixes, check the basename
    if img_id.startswith(("Dog Emotion Videos/", "Personal Data Set/")):
        basename = os.path.basename(img_id)
        for ext in ['.jpg', '.jpeg', '.png']:
            if os.path.exists(os.path.join(images_dir, f"{basename}{ext}")):
                filtered_annotations[img_id] = data
                found_count += 1
                break
    else:
        # Regular file check
        for ext in ['.jpg', '.jpeg', '.png']:
            if os.path.exists(os.path.join(images_dir, f"{img_id}{ext}")):
                filtered_annotations[img_id] = data
                found_count += 1
                break

# Save filtered annotations
os.makedirs(os.path.dirname(filtered_file), exist_ok=True)
with open(filtered_file, 'w') as f:
    json.dump(filtered_annotations, f, indent=2)

print(f"✅ Created filtered annotations with {found_count} entries out of {len(annotations)} total")